                                            Wojciech Olejnik
                                            
                                           "Selekcja zmiennych"
                                            
                                            Uczenie maszynowe 1

In [36]:
import pandas as pd
import numpy as np
from sklearn.datasets import fetch_rcv1
from sklearn.preprocessing import Binarizer
import time
from sklearn.datasets import fetch_rcv1

Fukcje do obliczania prawdopodobieństwa wystąpienia danej wartości, na podstawie którego będą liczone kolejne metryki :

In [37]:
# Zadanie 1
def freq(X, prob=True):
    xi, ni = np.unique(X)
    return xi, ni / np.sum(xi) if prob else np.sum(xi)

In [48]:
# Zadanie 2
def freq2(X, Y, prob=True):
    X = np.array(X)
    Y = np.array(Y)
    xi = np.unique(X)
    yj = np.unique(Y)
    nij = np.zeros((xi.size, yj.size), dtype='int32')
    for i, x in enumerate(xi):
        for j, y in enumerate(yj):
            nij[i, j] = np.sum(np.logical_and(X == x, Y == y))

    if prob:
        nij = nij/np.sum(nij)

    return xi, yj, nij

In [49]:
# Zadanie 3
def entropy(p):
    return -np.sum(p * np.log2(p+1e-16)).astype("float64")

def infogain(p):
    return entropy(np.sum(p, axis=1)) + entropy(np.sum(p, axis=0)) - entropy(p)

In [50]:
# Zadanie 4
def kappa(p):
    return infogain(p)/entropy(p).astype("float64")

In [51]:
# Zadanie 5
def gini(p):
    return 1 - np.sum(p**2)

def ginigain(p):
    GY = gini(np.sum(p, axis=0))
    PX = p.sum(axis=1, keepdims=True)
    GYX = np.sum(PX * (1 - np.sum((p/PX)**2, axis=1)))

    return GY - GYX

In [42]:
# Zadanie 6
df = pd.read_csv('./zoo.csv')
type = df['type'].to_list()

tag_name = {}
for i in range(1, 17):
    att = df.iloc[:, i].to_list()
    xi, yj, pij = freq2(att, type)
    k = kappa(pij)
    ig = infogain(pij)
    g = ginigain(pij)
    tag_name[df.columns[i]] = (k, g, ig)

rel = pd.DataFrame.from_dict(tag_name, orient='index', columns=['kappa', 'ginigain', 'infogain'])
print(rel)

             kappa  ginigain  infogain
hair      0.305999 -0.192399  0.790675
feathers  0.300327  0.072552  0.717950
eggs      0.326841 -0.117906  0.830138
milk      0.407570 -0.025663  0.974320
airborne  0.173199 -0.414579  0.469703
aquatic   0.132445 -0.654758  0.389487
predator  0.028416 -0.732882  0.093447
toothed   0.347187 -0.402903  0.865694
backbone  0.282847 -0.401931  0.676163
breathes  0.244472 -0.402502  0.614494
venomous  0.050093 -0.751014  0.133090
fins      0.181013 -0.320762  0.466614
legs      0.445247 -0.880952  1.363047
tail      0.184472 -0.635864  0.500460
domestic  0.017509 -0.575801  0.050669
catsize   0.100483 -0.519178  0.308490


Najbardziej istotna kolumna to : 'milk' lub 'legs'

In [43]:
    # Zadanie 8 - Matrix of sparse
    
    
    def freq_sparse_1(X):
        """ X - sparse of matrix """
        xi = [1, 0]
        count_ones = X.count_nonzero()
        true_proba = count_ones/X.shape[0]
        ni = [true_proba, 1-true_proba]
        
        return np.array(xi), np.array(ni)
            
    
    rcvl = fetch_rcv1()
    X = rcvl["data"]
    Y = rcvl.target[:, 87]
    

    xi, pi = freq_sparse_1(Y)
    print(f'xi:{xi}, pi:{pi}')
    # binarizer
#     binarizer = Binarizer().fit(X)
#     binary_X = binarizer.transform(X)
#     binarizer = Binarizer().fit(Y)
#     binary_Y = binarizer.transform(Y)
#     print(f'binary_X  : {binary_X}')
#     print(f'binary_Y  : {binary_Y}')

#     columns_name = {}
#     start = time.time()
#     for i in range(0, X.shape[1]):
#         attributes = binary_X[:, i]
#         # xi, yj, pij = freq2(attributes, Y)
#         k = kappa(pij)
#         ig = infogain(pij)
#         g = ginigain(pij)
#         columns_name[i] = (k, g, ig)

#     end = time.time()
#     relations = pd.DataFrame.from_dict(columns_name, orient='index', columns=['kappa', 'ginigain', 'infogain'])
#     print('Whole operation for ', binary_X.shape[1], 'words took: ', end - start)
#     print(relations)

xi:[1 0], pi:[0.04390401 0.95609599]


Przeprowadzono test funkcji freq_sparse_1 dla tag 'sport'.

In [ ]:
    def freq_sparse_2(X, Y, prob=True):
        xi = np.array([1, 0])
        x_count_ones = X.count_nonzero()
        x_counts = np.array([x_count_ones, X.shape[0]-x_count_ones])
        yj = np.array([1, 0])
        y_count_ones = Y.count_nonzero()
        y_counts = np.array([y_count_ones, Y.shape[0]-y_count_ones])
        x_ind = X.nonzero()[0]
        y_ind = Y.nonzero()[0]
        nij = np.zeros((xi.size, yj.size), dtype='int32')
        x_ind = X.nonzero()[0]
        y_ind = Y.nonzero()[0]
        x_set = set(x_ind)
        y_set = set(y_ind)
        x_and_y = x_set & y_set
        nij[1, 1] = len(x_and_y)
        nij[1, 0] = len(x_set - y_set)
        nij[0, 1] = len(y_set - x_set)
        nij[0, 0] = Y.shape[0] - (nij[1, 1]+nij[1, 0]+nij[0, 1])

        return xi, yj, nij
    
    
#     binarizer
    binarizer = Binarizer().fit(X)
    binary_X = binarizer.transform(X)
    binarizer = Binarizer().fit(Y)
    binary_Y = binarizer.transform(Y)

    columns_name = {}
    start = time.time()
    for i in range(0, X.shape[1]):
        attributes = binary_X[:, i]
        xi, yj, pij = freq_sparse_2(binary_X[:, 9], binary_Y)
    
    print(f'xi ; {xi}, yj ; {yj}, pij ; {pij}')